# Attention Is All You Need
**ArXivist-generated reproduction notebook**
Paper: Not provided
Generated: 2026-07-10

This notebook walks through the key components of the implementation, runs a
small-scale training loop, and verifies that the setup matches the paper's
reported behavior on a mini-dataset.

In [14]:
# Check Python version, GPU availability, and key dependencies
import sys, torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU — training will be slow")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Python: 3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:39:58) [MSC v.1943 64 bit (AMD64)]
PyTorch: 2.13.0+cpu
CUDA available: False
Running on CPU — training will be slow


In [15]:
# Install the project in editable mode (run once)
import subprocess
result = subprocess.run(["pip", "install", "-e", ".."], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)

Obtaining file:///F:/QOSI%20Fellowship/Research%20Papers/outputs/paper-repos/paper_attention-is-all-you-need
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for transformer (pyproject.toml): started
  Building editable for transformer (pyproject.toml): finished with status 'done'
  Created wheel for transformer: filename=transformer-0.1.0-0.editable-py3-none-any.whl size=1387 sha256=fef29a97ac1f0065c1d12f4bc29a20399f9c64fa4607b3f1259fae4490d80be3
  Stored in directory: C:\Users\Mehansh\AppData\Local\Temp\pip-ephem-wheel-cache-

## Paper Overview

**Problem**: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. 

**Core Idea**: We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality.

**Mapping to implementation**:
- `src.transformer.models.attention`: Implements Scaled Dot-Product Attention (3.2.1) and Multi-Head Attention (3.2.2).
- `src.transformer.models.layers`: Implements Position-wise Feed-Forward Network (3.3) and Positional Encoding (3.5).
- `src.transformer.models.transformer`: Implements Encoder Stack (3.1), Decoder Stack (3.1), and the full Transformer model.

## Positional Encoding
Since the Transformer has no recurrence or convolution, it must inject some information about the relative or absolute position of the tokens in the sequence. This is done using sine and cosine functions of different frequencies.
$$ PE_{(pos, 2i)} = \sin(pos/10000^{2i/d_{\text{model}}}) $$
$$ PE_{(pos, 2i+1)} = \cos(pos/10000^{2i/d_{\text{model}}}) $$

In [16]:
import torch

try:
    from src.transformer.models.layers import PositionalEncoding
    d_model = 512
    max_len = 100
    pos_encoder = PositionalEncoding(d_model=d_model, max_len=max_len).to(device)
    
    x = torch.zeros(2, 10, d_model).to(device) # Batch size 2, Sequence length 10
    output = pos_encoder(x)
    print(f"Input shape:  {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Expected:     torch.Size([2, 10, {d_model}])")
except Exception as e:
    print(f"Error: {e}")

Input shape:  torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])
Expected:     torch.Size([2, 10, 512])


## Multi-Head Attention
Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions.
$$ \text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O $$
$$ \text{Attention}(Q, K, V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V $$

In [17]:
import torch

try:
    from src.transformer.models.attention import MultiHeadAttention
    model_config = {
        'h': 8,
        'd_model': 512,
        'd_k': 64,
        'd_v': 64,
        'dropout': 0.1
    }
    attention = MultiHeadAttention(**model_config).to(device)
    
    # Toy forward pass (Self-Attention)
    q = torch.randn(2, 10, 512).to(device)
    k = q
    v = q
    
    output = attention(q, k, v)
    print(f"Input shape:  {q.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Expected:     torch.Size([2, 10, 512])")
except Exception as e:
    print(f"Error: {e}")

Input shape:  torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])
Expected:     torch.Size([2, 10, 512])


## Position-wise Feed-Forward Network
In addition to attention sub-layers, each of the layers in our encoder and decoder contains a fully connected feed-forward network, which is applied to each position separately and identically.
$$ \text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2 $$

In [18]:
import torch

try:
    from src.transformer.models.layers import PositionwiseFeedForward
    model_config = {
        'd_model': 512,
        'd_ff': 2048,
        'dropout': 0.1
    }
    ffn = PositionwiseFeedForward(**model_config).to(device)
    
    x = torch.randn(2, 10, 512).to(device)
    output = ffn(x)
    print(f"Input shape:  {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Expected:     torch.Size([2, 10, 512])")
except Exception as e:
    print(f"Error: {e}")

Input shape:  torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])
Expected:     torch.Size([2, 10, 512])


## Encoder Stack
The encoder is composed of a stack of $N=6$ identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-wise fully connected feed-forward network. We employ a residual connection around each of the two sub-layers, followed by layer normalization.

In [19]:
import torch

try:
    from src.transformer.models.transformer import make_model
    model = make_model(src_vocab=100, tgt_vocab=100)
    encoder = model.encoder.to(device)

    # Pass embedded tensors instead of raw tokens!
    x = torch.randn(2, 10, 512).to(device)
    mask = torch.ones(2, 1, 1, 10).to(device)

    output = encoder(x, mask)
    print(f"Input shape:  {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Expected:     torch.Size([2, 10, 512])")
except Exception as e:
    print(f"Error: {e}")

Input shape:  torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])
Expected:     torch.Size([2, 10, 512])


## Decoder Stack
The decoder is also composed of a stack of $N=6$ identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack.

In [20]:
import torch

try:
    from src.transformer.models.transformer import make_model
    model = make_model(src_vocab=100, tgt_vocab=100)
    decoder = model.decoder.to(device)

    # Pass embedded tensors instead of raw tokens!
    tgt = torch.randn(2, 10, 512).to(device)
    memory = torch.randn(2, 10, 512).to(device)
    tgt_mask = torch.ones(2, 1, 10, 10).to(device)
    memory_mask = torch.ones(2, 1, 10, 10).to(device)

    output = decoder(tgt, memory, memory_mask, tgt_mask)
    print(f"Input shape:  {tgt.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Expected:     torch.Size([2, 10, 512])")
except Exception as e:
    print(f"Error: {e}")

Input shape:  torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])
Expected:     torch.Size([2, 10, 512])


## Mini-Training Demonstration

In [21]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Generate a tiny synthetic dataset for seq2seq (e.g., copying task)
vocab_size = 100
seq_length = 15
num_samples = 100

src_data = torch.randint(1, vocab_size, (num_samples, seq_length))
tgt_data = src_data.clone() # Simple copy task for demonstration

dataset = TensorDataset(src_data, tgt_data)
dataloader = DataLoader(dataset, batch_size=10, shuffle=True)
print(f"Dataset created with {num_samples} samples.")

Dataset created with 100 samples.


In [22]:
import torch.nn as nn
import torch.optim as optim

try:
    from src.transformer.models.transformer import make_model
    model_config = {
        'N': 2, # Reduced for speed
        'd_model': 128,
        'd_ff': 256,
        'h': 4,
        'd_k': 32,
        'd_v': 32,
        'dropout': 0.1,
        'src_vocab': vocab_size,
        'tgt_vocab': vocab_size
    }
    # We use make_model here instead of Transformer directly!
    model = make_model(**model_config).to(device)

    # Print parameter count
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model instantiated with {total_params:,} trainable parameters.")

    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
except Exception as e:
    print(f"Error: {e}")

Model instantiated with 701,540 trainable parameters.


In [23]:
try:
    model.train()
    epochs = 5
    print("Starting mini-training loop...")
    for epoch in range(epochs):
        epoch_loss = 0.0
        for src, tgt in dataloader:
            src = src.to(device)
            tgt = tgt.to(device)
            
            # For seq2seq, tgt_input is tgt[:-1], tgt_output is tgt[1:]
            tgt_input = tgt[:, :-1]
            tgt_expected = tgt[:, 1:]
            
            # Mock masks
            src_mask = (src != 0).unsqueeze(1).unsqueeze(2).to(device)
            tgt_mask = torch.tril(torch.ones((tgt_input.size(1), tgt_input.size(1)))).bool().to(device)
            
            optimizer.zero_grad()
            output = model(src, tgt_input, src_mask, tgt_mask)
            
            # output shape: [B, T, vocab_size]
            loss = criterion(output.reshape(-1, vocab_size), tgt_expected.reshape(-1))
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss/len(dataloader):.4f}")
    print("Mini-training complete.")
except Exception as e:
    print(f"Error: {e}")

Starting mini-training loop...
Epoch 1/5 - Loss: 4.9285
Epoch 2/5 - Loss: 4.5371
Epoch 3/5 - Loss: 4.3526
Epoch 4/5 - Loss: 3.9374
Epoch 5/5 - Loss: 3.4514
Mini-training complete.


In [24]:
try:
    model.eval()
    with torch.no_grad():
        sample_src = src_data[0:1].to(device)
        sample_tgt = sample_src.clone()[:, :-1] # Teacher forcing for demo
        src_mask = (sample_src != 0).unsqueeze(1).unsqueeze(2).to(device)
        tgt_mask = torch.tril(torch.ones((sample_tgt.size(1), sample_tgt.size(1)))).bool().to(device)
        
        output = model(sample_src, sample_tgt, src_mask, tgt_mask)
        predicted_tokens = output.argmax(dim=-1)
        
        print(f"Source token:   {sample_src[0].cpu().tolist()}")
        print(f"Predicted next: {predicted_tokens[0].cpu().tolist()}")
except Exception as e:
    print(f"Error: {e}")

Source token:   [35, 88, 59, 68, 66, 84, 54, 64, 89, 10, 26, 21, 18, 19, 91]
Predicted next: [78, 59, 66, 66, 66, 66, 78, 89, 54, 54, 54, 54, 84, 54]


## Paper Results Comparison

In [25]:
# Results reported in the paper (from SIR evaluation_protocol.reported_results)
paper_results = [
    {
        "dataset": "WMT 2014 English-German",
        "metric": "BLEU",
        "reported_value": 28.4
    },
    {
        "dataset": "WMT 2014 English-French",
        "metric": "BLEU",
        "reported_value": 41.0
    }
]
print("Paper's claimed results:")
for res in paper_results:
    print(f"  {res['dataset']} - {res['metric']}: {res['reported_value']}")
print("\nTo reproduce these results, run train.py with the full config.")
print("Then use the Results Comparator (Stage 6) to compare your outputs.")

Paper's claimed results:
  WMT 2014 English-German - BLEU: 28.4
  WMT 2014 English-French - BLEU: 41.0

To reproduce these results, run train.py with the full config.
Then use the Results Comparator (Stage 6) to compare your outputs.


## What to do next

1. **Full training**: `python train.py --config configs/config.yaml`
2. **Evaluation**: `python evaluate.py --checkpoint checkpoints/best.pt`
3. **Compare results**: Feed your results back to ArXivist's Results Comparator

**Implementation notes from the SIR:**
- Batch dimension is the first dimension in tensors (e.g., [B, T, D]) (Confidence: 0.9)
- Weights are initialized using standard initialization schemes like Xavier/Glorot. (Confidence: 0.8)
- Masking out illegal connections sets values to -infinity. Implementation in practice uses a very large negative number (-1e9). (Confidence: 0.85)